# 2D Global Map Visualization for ROCKE-3D Outputs

This Jupyter Notebook visualizes the **single-layer global pixel map outputs** generated by the ROCKE-3D JAX and Fortran workflows. It projects the outputs onto **2D global maps** (latitude vs. longitude) for easier interpretation.

## Outputs Visualized:
- **Temperature (K)**
- **Pressure (hPa)**
- **Heat Flux (W/m²)**
- **Solar Flux (W/m²)**

## Implementations Compared:
- **JAX (Python)**
- **Fortran**

## Requirements:
- `numpy`
- `plotly`

## Usage:
1. Run the notebook to load and visualize the outputs from the `outputs/` directory and Fortran binary files.
2. Interact with the **2D global maps** to explore the data.

## 1. Import Required Libraries

Import the necessary libraries for loading data and creating interactive 2D visualizations.

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load Data

Load the single-layer global pixel map outputs from the `outputs/` directory (JAX) and Fortran binary files.

In [2]:
# Constants for grid dimensions
N_LAT = 180
N_LON = 360

# Load JAX outputs from .npy files
lat = np.load('outputs/lat.npy')
lon = np.load('outputs/lon.npy')

temperature_jax = np.load('outputs/temperature.npy')
pressure_jax = np.load('outputs/pressure.npy')
heat_flux_jax = np.load('outputs/heat_flux.npy')
solar_flux_jax = np.load('outputs/solar_flux.npy')

print(f"JAX Temperature shape: {temperature_jax.shape}")
print(f"JAX Temperature first row (lat=-90): {temperature_jax[0, :5]}")
print(f"JAX Temperature last row (lat=90): {temperature_jax[-1, :5]}")

# Function to read Fortran unformatted binary files
def read_fortran_binary(filename, shape):
    """Read a Fortran unformatted binary file."""
    with open(filename, 'rb') as f:
        f.read(4)  # Skip the first 4 bytes (record length)
        data = np.fromfile(f, dtype=np.float64)
        f.read(4)  # Skip the last 4 bytes (record length)
    return data.reshape(shape, order='F')  # Use Fortran order for reshaping

# Load Fortran outputs from binary files
fortran_base_path = '/home/gtamkin/_ilab-agentic-ai/ilab-agentic-ai/projects/imvi/modelE2_planet_2.0/model'

# Read Fortran data in column-major order and convert to row-major
temperature_fortran = read_fortran_binary(
    f'{fortran_base_path}/fortran_temperature.bin',
    (N_LAT, N_LON)
)

pressure_fortran = read_fortran_binary(
    f'{fortran_base_path}/fortran_pressure.bin',
    (N_LAT, N_LON)
)

heat_flux_fortran = read_fortran_binary(
    f'{fortran_base_path}/fortran_heat_flux.bin',
    (N_LAT, N_LON)
)

solar_flux_fortran = read_fortran_binary(
    f'{fortran_base_path}/fortran_solar_flux.bin',
    (N_LAT, N_LON)
)

print(f"\nFortran Temperature shape: {temperature_fortran.shape}")
print(f"Fortran Temperature first row (lat=-90): {temperature_fortran[0, :5]}")
print(f"Fortran Temperature last row (lat=90): {temperature_fortran[-1, :5]}")

# Verify alignment with JAX
print(f"\nJAX first row matches Fortran first row: {np.allclose(temperature_jax[0, :], temperature_fortran[0, :])}")
print(f"JAX last row matches Fortran last row: {np.allclose(temperature_jax[-1, :], temperature_fortran[-1, :])}")

print("\nData loaded successfully!")

JAX Temperature shape: (180, 360)
JAX Temperature first row (lat=-90): [268. 268. 268. 268. 268.]
JAX Temperature last row (lat=90): [308. 308. 308. 308. 308.]


FileNotFoundError: [Errno 2] No such file or directory: '/home/gtamkin/_ilab-agentic-ai/ilab-agentic-ai/projects/imvi/modelE2_planet_2.0/model/fortran_temperature.bin'

## 3. Create 2D Global Maps

Create **2D global maps** (latitude vs. longitude) for each variable and implementation.

In [ ]:
def create_2d_global_map(data, lat, lon, title, colorscale='Viridis'):
    """
    Create a 2D global map (latitude vs. longitude) using Plotly.
    
    Args:
        data: 2D array of shape (n_lat, n_lon)
        lat: Latitude array
        lon: Longitude array
        title: Plot title
        colorscale: Color scale for the plot
    """
    fig = go.Figure(data=go.Heatmap(
        z=data,
        x=lon,
        y=lat,
        colorscale=colorscale,
        colorbar=dict(title=title.split(':')[1].strip() if ':' in title else title),
        hoverongaps=False
    ))
    
    fig.update_layout(
        title=title,
        xaxis_title='Longitude',
        yaxis_title='Latitude',
        width=800,
        height=400
    )
    
    return fig

# Create 2D global maps for JAX outputs
temp_jax_fig = create_2d_global_map(temperature_jax, lat, lon, "JAX: Global Temperature (K)", "RdYlBu_r")
pres_jax_fig = create_2d_global_map(pressure_jax, lat, lon, "JAX: Global Pressure (hPa)", "Blues")
heat_jax_fig = create_2d_global_map(heat_flux_jax, lat, lon, "JAX: Global Heat Flux (W/m²)", "Reds")
solar_jax_fig = create_2d_global_map(solar_flux_jax, lat, lon, "JAX: Global Solar Flux (W/m²)", "YlOrRd")

# Display JAX plots
temp_jax_fig.show()
pres_jax_fig.show()
heat_jax_fig.show()
solar_jax_fig.show()

# Create 2D global maps for Fortran outputs
temp_fortran_fig = create_2d_global_map(temperature_fortran, lat, lon, "Fortran: Global Temperature (K)", "RdYlBu_r")
pres_fortran_fig = create_2d_global_map(pressure_fortran, lat, lon, "Fortran: Global Pressure (hPa)", "Blues")
heat_fortran_fig = create_2d_global_map(heat_flux_fortran, lat, lon, "Fortran: Global Heat Flux (W/m²)", "Reds")
solar_fortran_fig = create_2d_global_map(solar_flux_fortran, lat, lon, "Fortran: Global Solar Flux (W/m²)", "YlOrRd")

# Display Fortran plots
temp_fortran_fig.show()
pres_fortran_fig.show()
heat_fortran_fig.show()
solar_fortran_fig.show()

## 5. Difference Maps (Fortran - JAX)

Create 2D global maps showing the **pixel-wise differences** between Fortran and JAX outputs.

In [ ]:
# Compute pixel-wise differences (Fortran - JAX)
temp_diff = temperature_fortran - temperature_jax
pres_diff = pressure_fortran - pressure_jax
heat_diff = heat_flux_fortran - heat_flux_jax
solar_diff = solar_flux_fortran - solar_flux_jax

# Create 2D global maps for the differences
temp_diff_fig = create_2d_global_map(temp_diff, lat, lon, "Difference: Temperature (K)", "RdYlBu")
pres_diff_fig = create_2d_global_map(pres_diff, lat, lon, "Difference: Pressure (hPa)", "RdYlBu")
heat_diff_fig = create_2d_global_map(heat_diff, lat, lon, "Difference: Heat Flux (W/m²)", "RdYlBu")
solar_diff_fig = create_2d_global_map(solar_diff, lat, lon, "Difference: Solar Flux (W/m²)", "RdYlBu")

# Display difference maps
temp_diff_fig.show()
pres_diff_fig.show()
heat_diff_fig.show()
solar_diff_fig.show()

# Print summary statistics for differences
print("\nDifference Statistics:")
print(f"Temperature (K) - Min: {temp_diff.min():.6f}, Max: {temp_diff.max():.6f}, Mean: {temp_diff.mean():.6f}")
print(f"Pressure (hPa) - Min: {pres_diff.min():.6f}, Max: {pres_diff.max():.6f}, Mean: {pres_diff.mean():.6f}")
print(f"Heat Flux (W/m²) - Min: {heat_diff.min():.6f}, Max: {heat_diff.max():.6f}, Mean: {heat_diff.mean():.6f}")
print(f"Solar Flux (W/m²) - Min: {solar_diff.min():.6f}, Max: {solar_diff.max():.6f}, Mean: {solar_diff.mean():.6f}")

## 6. Combined Dashboard for Differences

Create a **combined dashboard** with all difference maps.

In [ ]:
# Create a combined dashboard for difference maps
fig_diff = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Difference: Temperature (K)",
        "Difference: Pressure (hPa)",
        "Difference: Heat Flux (W/m²)",
        "Difference: Solar Flux (W/m²)"
    ),
    horizontal_spacing=0.1,
    vertical_spacing=0.1
)

# Add difference maps to the dashboard
fig_diff.add_trace(
    go.Heatmap(z=temp_diff, x=lon, y=lat, colorscale="RdYlBu", showscale=True),
    row=1, col=1
)

fig_diff.add_trace(
    go.Heatmap(z=pres_diff, x=lon, y=lat, colorscale="RdYlBu", showscale=True),
    row=1, col=2
)

fig_diff.add_trace(
    go.Heatmap(z=heat_diff, x=lon, y=lat, colorscale="RdYlBu", showscale=True),
    row=2, col=1
)

fig_diff.add_trace(
    go.Heatmap(z=solar_diff, x=lon, y=lat, colorscale="RdYlBu", showscale=True),
    row=2, col=2
)

# Update layout
fig_diff.update_layout(
    title="Pixel-Wise Differences: Fortran - JAX",
    width=1000,
    height=800
)

# Update axes
for i in range(1, 3):
    fig_diff.update_xaxes(title_text="Longitude", row=i, col=1)
    fig_diff.update_xaxes(title_text="Longitude", row=i, col=2)
    fig_diff.update_yaxes(title_text="Latitude", row=i, col=1)
    fig_diff.update_yaxes(title_text="Latitude", row=i, col=2)

# Display the dashboard
fig_diff.show()

## 7. Save Difference Visualizations

Save all difference maps as HTML files for offline viewing.

In [ ]:
# Compute pixel-wise differences (Fortran - JAX)
# Round to 3 decimal places to remove floating-point noise
temp_diff = np.round(temperature_fortran - temperature_jax, 3)
pres_diff = np.round(pressure_fortran - pressure_jax, 3)
heat_diff = np.round(heat_flux_fortran - heat_flux_jax, 3)
solar_diff = np.round(solar_flux_fortran - solar_flux_jax, 3)

# Create 2D global maps for the differences
# Use a discrete color scale to highlight only non-zero differences
# White for zero, red for positive, blue for negative
temp_diff_fig = create_2d_global_map(temp_diff, lat, lon, "Difference: Temperature (K)", "RdYlBu")
pres_diff_fig = create_2d_global_map(pres_diff, lat, lon, "Difference: Pressure (hPa)", "RdYlBu")
heat_diff_fig = create_2d_global_map(heat_diff, lat, lon, "Difference: Heat Flux (W/m²)", "RdYlBu")
solar_diff_fig = create_2d_global_map(solar_diff, lat, lon, "Difference: Solar Flux (W/m²)", "RdYlBu")

# Display difference maps
temp_diff_fig.show()
pres_diff_fig.show()
heat_diff_fig.show()
solar_diff_fig.show()

# Print summary statistics for differences
print("\nDifference Statistics:")
print(f"Temperature (K) - Min: {temp_diff.min():.6f}, Max: {temp_diff.max():.6f}, Mean: {temp_diff.mean():.6f}")
print(f"Pressure (hPa) - Min: {pres_diff.min():.6f}, Max: {pres_diff.max():.6f}, Mean: {pres_diff.mean():.6f}")
print(f"Heat Flux (W/m²) - Min: {heat_diff.min():.6f}, Max: {heat_diff.max():.6f}, Mean: {heat_diff.mean():.6f}")
print(f"Solar Flux (W/m²) - Min: {solar_diff.min():.6f}, Max: {solar_diff.max():.6f}, Mean: {solar_diff.mean():.6f}")

# Count the number of non-zero differences
print(f"\nNon-zero differences:")
print(f"Temperature: {np.count_nonzero(temp_diff)} pixels")
print(f"Pressure: {np.count_nonzero(pres_diff)} pixels")
print(f"Heat Flux: {np.count_nonzero(heat_diff)} pixels")
print(f"Solar Flux: {np.count_nonzero(solar_diff)} pixels")

## 4. Combined Dashboard for JAX and Fortran

Create a **combined dashboard** with 2D global maps for both JAX and Fortran outputs.

In [ ]:
# Create a combined dashboard with 2D global maps for JAX, Fortran, and Differences
fig = make_subplots(
    rows=4, cols=3,
    subplot_titles=(
        "JAX: Temperature (K)",
        "Fortran: Temperature (K)",
        "Difference: Temperature (K)",
        "JAX: Pressure (hPa)",
        "Fortran: Pressure (hPa)",
        "Difference: Pressure (hPa)",
        "JAX: Heat Flux (W/m²)",
        "Fortran: Heat Flux (W/m²)",
        "Difference: Heat Flux (W/m²)",
        "JAX: Solar Flux (W/m²)",
        "Fortran: Solar Flux (W/m²)",
        "Difference: Solar Flux (W/m²)"
    ),
    horizontal_spacing=0.1,
    vertical_spacing=0.1
)

# Add JAX, Fortran, and Difference data to the dashboard
# Temperature
fig.add_trace(
    go.Heatmap(z=temperature_jax, x=lon, y=lat, colorscale="RdYlBu_r", showscale=False),
    row=1, col=1
)
fig.add_trace(
    go.Heatmap(z=temperature_fortran, x=lon, y=lat, colorscale="RdYlBu_r", showscale=False),
    row=1, col=2
)
fig.add_trace(
    go.Heatmap(z=temp_diff, x=lon, y=lat, colorscale="RdYlBu", showscale=True, zmin=-1e-3, zmax=1e-3, zmid=0),
    row=1, col=3
)

# Pressure
fig.add_trace(
    go.Heatmap(z=pressure_jax, x=lon, y=lat, colorscale="Blues", showscale=False),
    row=2, col=1
)
fig.add_trace(
    go.Heatmap(z=pressure_fortran, x=lon, y=lat, colorscale="Blues", showscale=False),
    row=2, col=2
)
fig.add_trace(
    go.Heatmap(z=pres_diff, x=lon, y=lat, colorscale="RdYlBu", showscale=True, zmin=-1e-3, zmax=1e-3, zmid=0),
    row=2, col=3
)

# Heat Flux
fig.add_trace(
    go.Heatmap(z=heat_flux_jax, x=lon, y=lat, colorscale="Reds", showscale=False),
    row=3, col=1
)
fig.add_trace(
    go.Heatmap(z=heat_flux_fortran, x=lon, y=lat, colorscale="Reds", showscale=False),
    row=3, col=2
)
fig.add_trace(
    go.Heatmap(z=heat_diff, x=lon, y=lat, colorscale="RdYlBu", showscale=True, zmin=-1e-3, zmax=1e-3, zmid=0),
    row=3, col=3
)

# Solar Flux
fig.add_trace(
    go.Heatmap(z=solar_flux_jax, x=lon, y=lat, colorscale="YlOrRd", showscale=False),
    row=4, col=1
)
fig.add_trace(
    go.Heatmap(z=solar_flux_fortran, x=lon, y=lat, colorscale="YlOrRd", showscale=False),
    row=4, col=2
)
fig.add_trace(
    go.Heatmap(z=solar_diff, x=lon, y=lat, colorscale="RdYlBu", showscale=True, zmin=-1e-3, zmax=1e-3, zmid=0),
    row=4, col=3
)

# Update layout
fig.update_layout(
    title="2D Global Maps: JAX vs. Fortran vs. Differences",
    width=1500,
    height=1200
)

# Update axes
for i in range(1, 5):
    fig.update_xaxes(title_text="Longitude", row=i, col=1)
    fig.update_xaxes(title_text="Longitude", row=i, col=2)
    fig.update_xaxes(title_text="Longitude", row=i, col=3)
    fig.update_yaxes(title_text="Latitude", row=i, col=1)
    fig.update_yaxes(title_text="Latitude", row=i, col=2)
    fig.update_yaxes(title_text="Latitude", row=i, col=3)

# Display the dashboard
fig.show()

## 5. Save Visualizations

Save all 2D global maps as HTML files for offline viewing.

In [ ]:
# Save individual 2D global maps as HTML files
temp_jax_fig.write_html("outputs/jax_temperature_2d_map.html")
pres_jax_fig.write_html("outputs/jax_pressure_2d_map.html")
heat_jax_fig.write_html("outputs/jax_heat_flux_2d_map.html")
solar_jax_fig.write_html("outputs/jax_solar_flux_2d_map.html")

temp_fortran_fig.write_html("outputs/fortran_temperature_2d_map.html")
pres_fortran_fig.write_html("outputs/fortran_pressure_2d_map.html")
heat_fortran_fig.write_html("outputs/fortran_heat_flux_2d_map.html")
solar_fortran_fig.write_html("outputs/fortran_solar_flux_2d_map.html")

# Save the combined dashboard
fig.write_html("outputs/2d_global_maps_dashboard.html")

print("\n2D Global Map visualizations saved successfully in the 'outputs/' directory!")
print("\nSaved files:")
print("- JAX: temperature_2d_map.html, pressure_2d_map.html, heat_flux_2d_map.html, solar_flux_2d_map.html")
print("- Fortran: temperature_2d_map.html, pressure_2d_map.html, heat_flux_2d_map.html, solar_flux_2d_map.html")
print("- Combined: 2d_global_maps_dashboard.html")